# BAA10Y Adaptive Agent Test


In [1]:
from pathlib import Path
import json
import warnings
warnings.filterwarnings('ignore')

from BAA10Y_forecasting.adaptive_agent.tuner import (
    BAA10YTuner,
)
import pandas as pd
from IPython.display import display
from BAA10Y_forecasting.predictors.benchmark_configs import (
    NOTEBOOK_01_BENCHMARKS,
    get_benchmark_config,
)


In [2]:
def find_repo_root() -> Path:
    """Find the agentic-forecasting repository root."""

    here = Path.cwd().resolve()

    for candidate in (
        here,
        *here.parents,
    ):
        if (
            (
                candidate
                / "pyproject.toml"
            ).exists()
            and (
                candidate
                / "aieng-forecasting"
            ).is_dir()
        ):
            return candidate

    raise RuntimeError(
        "Could not find repository root."
    )


CHECKS = []


def add_check(
    category: str,
    name: str,
    passed: bool,
    details: str = "",
) -> None:
    """Add one result to the notebook health report."""

    CHECKS.append({
        "category": category,
        "check": name,
        "status": (
            "PASS"
            if passed
            else "NOT YET"
        ),
        "details": details,
    })


ROOT = find_repo_root()

# This directory separates the Notebook 04 LightGBM
# demonstration from older smoke-test and tuning state.
#
# To start a completely new study later, change v1 to v2.
DEMO_RUN_ID = (
    "lightgbm_h5_default_v1"
)

STATE_PATH = (
    ROOT
    / "implementations"
    / "BAA10Y_forecasting"
    / "adaptive_agent"
    / "state"
    / DEMO_RUN_ID
    / "tuning_state.yaml"
)


print("\nAdaptive LightGBM state:")
print(STATE_PATH)


Adaptive LightGBM state:
/home/coder/agentic-forecasting/implementations/BAA10Y_forecasting/adaptive_agent/state/lightgbm_h5_default_v1/tuning_state.yaml


## 1. Register and verify adaptive-agent tools

In [10]:
from BAA10Y_forecasting.adaptive_agent.agent import (
    build_baa10y_adaptive_config,
)
from BAA10Y_forecasting.adaptive_agent.skill_state import (
    TuningStateStore,
)
from BAA10Y_forecasting.adaptive_agent.skill_tools import (
    build_baa10y_tuning_tools,
)
from BAA10Y_forecasting.adaptive_agent.tuner import (
    BAA10YTuner,
)


add_check(
    "technical",
    "Adaptive modules import",
    True,
)

TOOLS = build_baa10y_tuning_tools(
    state_path=STATE_PATH,
)

TOOL_BY_NAME = {
    tool.__name__: tool
    for tool in TOOLS
}

EXPECTED_TOOLS = {
    "get_tuning_state",
    "list_tuning_candidates",
    "get_search_space",
    "run_tuning_trial",
    "run_adaptive_search",
    "get_search_diagnostics",
    "freeze_search_candidate",
    "run_frozen_validation",
    "compare_tuning_trials",
    "promote_tuning_candidate",
    "reject_tuning_candidate",
}

tool_names = set(
    TOOL_BY_NAME
)

missing_tools = (
    EXPECTED_TOOLS
    - tool_names
)

tools_ok = not missing_tools

add_check(
    "technical",
    "Adaptive tuning tools registered",
    tools_ok,
    str(sorted(tool_names)),
)

print(
    "Registered adaptive-agent tools:"
)

for name in sorted(tool_names):
    print(" -", name)

assert tools_ok, (
    "Missing adaptive-agent tools: "
    f"{sorted(missing_tools)}"
)

Registered adaptive-agent tools:
 - compare_tuning_trials
 - freeze_search_candidate
 - get_search_diagnostics
 - get_search_space
 - get_tuning_state
 - list_tuning_candidates
 - promote_tuning_candidate
 - reject_tuning_candidate
 - run_adaptive_search
 - run_frozen_validation
 - run_tuning_trial


## 2. Configure the LightGBM adaptive test

In [11]:
import json

from BAA10Y_forecasting.predictors.benchmark_configs import (
    get_benchmark_config,
)


def call_json_tool(
    tool_name: str,
    **kwargs,
) -> dict:
    """Call one registered agent tool and parse its JSON response."""

    if tool_name not in TOOL_BY_NAME:
        raise KeyError(
            f"Unknown tool {tool_name!r}. "
            f"Available tools: "
            f"{sorted(TOOL_BY_NAME)}"
        )

    raw_output = (
        TOOL_BY_NAME[tool_name](
            **kwargs
        )
    )

    payload = json.loads(
        raw_output
    )

    if payload.get("status") == "error":
        raise RuntimeError(
            payload.get(
                "message",
                payload,
            )
        )

    return payload


METHOD = "lightgbm"
HORIZON = 5
COVARIATE_PANEL = "default"

INITIAL_TRIALS = 5
TOTAL_TRIALS = 10


BENCHMARK_PARAMETERS = (
    get_benchmark_config(
        METHOD,
        COVARIATE_PANEL,
    )
)

SEARCH_SPACE = call_json_tool(
    "get_search_space",
    method=METHOD,
)


search_space_ok = (
    SEARCH_SPACE["method"]
    == METHOD
    and SEARCH_SPACE["strategy"]
    == "optuna_tpe"
    and COVARIATE_PANEL
    in SEARCH_SPACE[
        "covariate_panels"
    ]
)

add_check(
    "technical",
    "LightGBM search space available",
    search_space_ok,
    (
        f"strategy="
        f"{SEARCH_SPACE['strategy']}; "
        f"panel={COVARIATE_PANEL}"
    ),
)

assert search_space_ok


print("Adaptive test configuration:")
print(" Method:", METHOD)
print(" Horizon:", HORIZON)
print(
    " Covariate panel:",
    COVARIATE_PANEL,
)
print(
    " Initial trials:",
    INITIAL_TRIALS,
)
print(
    " Total trials:",
    TOTAL_TRIALS,
)

print("\nMatching baseline parameters:")
display(BENCHMARK_PARAMETERS)

print("\nApproved LightGBM search space:")
display(
    SEARCH_SPACE[
        "parameter_space"
    ]
)

print("\nAllowed agent focus actions:")
display(
    SEARCH_SPACE[
        "focus_actions"
    ]
)

Adaptive test configuration:
 Method: lightgbm
 Horizon: 5
 Covariate panel: default
 Initial trials: 5
 Total trials: 10

Matching baseline parameters:


{'lags': 5,
 'lags_past_covariates': 5,
 'num_samples': 100,
 'lgbm_kwargs': {'num_threads': 1,
  'n_jobs': 1,
  'verbosity': -1,
  'random_state': 42}}


Approved LightGBM search space:


{'lags': [3, 5, 10, 21],
 'lags_past_covariates': [3, 5, 10, 21],
 'n_estimators': {'minimum': 50, 'maximum': 400, 'step': 50},
 'learning_rate': {'minimum': 0.02, 'maximum': 0.15, 'scale': 'log'},
 'tree_shapes': [{'max_depth': 3, 'num_leaves': 7},
  {'max_depth': 4, 'num_leaves': 15},
  {'max_depth': 6, 'num_leaves': 31},
  {'max_depth': -1, 'num_leaves': 15},
  {'max_depth': -1, 'num_leaves': 31}],
 'min_child_samples': [10, 20, 40],
 'reg_alpha_l1': {'minimum': 0.0, 'maximum': 2.0},
 'reg_lambda_l2': {'minimum': 0.0, 'maximum': 5.0},
 'subsample': [0.7, 0.85, 1.0],
 'colsample_bytree': [0.7, 0.85, 1.0]}


Allowed agent focus actions:


['broad_search',
 'regularize_more',
 'reduce_complexity',
 'stabilize_boosting',
 'continue_tpe']

In [13]:
import yaml

from aieng.forecasting.evaluation import (
    MultiTargetBacktestSpec,
)


SPEC_DIRECTORY = (
    ROOT
    / "implementations"
    / "BAA10Y_forecasting"
    / "specs"
)

EXPERIMENT_SPECS = {
    "development": (
        "baa10y_tune_development_2024.yaml"
    ),
    "inner_validation": (
        "baa10y_tune_inner_validation_2025.yaml"
    ),
    "outer_validation": (
        "baa10y_validate_2025.yaml"
    ),
}


def load_backtest_spec(
    filename: str,
) -> MultiTargetBacktestSpec:
    """Load and validate one backtest specification."""

    path = (
        SPEC_DIRECTORY
        / filename
    )

    with path.open(
        encoding="utf-8"
    ) as file:
        contents = yaml.safe_load(
            file
        )

    return (
        MultiTargetBacktestSpec
        .model_validate(contents)
    )


LOADED_SPECS = {
    name: load_backtest_spec(
        filename
    )
    for name, filename
    in EXPERIMENT_SPECS.items()
}


period_rows = []

for name, specification in (
    LOADED_SPECS.items()
):
    start = pd.Timestamp(
        specification.start
    )

    end = pd.Timestamp(
        specification.end
    )

    origins = pd.date_range(
        start=start,
        end=end,
        freq="B",
    )[::specification.stride]

    period_rows.append({
        "period": name,
        "start": start.date(),
        "end": end.date(),
        "stride": specification.stride,
        "number_of_origins": (
            len(origins)
        ),
        "first_origin": (
            origins.min().date()
        ),
        "last_origin": (
            origins.max().date()
        ),
    })


EVALUATION_PERIODS = pd.DataFrame(
    period_rows
)

display(EVALUATION_PERIODS)


development_end = pd.Timestamp(
    LOADED_SPECS[
        "development"
    ].end
)

inner_validation_start = pd.Timestamp(
    LOADED_SPECS[
        "inner_validation"
    ].start
)

inner_validation_end = pd.Timestamp(
    LOADED_SPECS[
        "inner_validation"
    ].end
)

outer_validation_start = pd.Timestamp(
    LOADED_SPECS[
        "outer_validation"
    ].start
)


periods_separated = (
    development_end
    < inner_validation_start
    and inner_validation_end
    < outer_validation_start
)

add_check(
    "governance",
    "Tuning and validation periods separated",
    periods_separated,
    (
        f"development_end="
        f"{development_end.date()}; "
        f"inner_start="
        f"{inner_validation_start.date()}; "
        f"inner_end="
        f"{inner_validation_end.date()}; "
        f"outer_start="
        f"{outer_validation_start.date()}"
    ),
)

assert periods_separated, (
    "Development, inner-validation and "
    "outer-validation periods overlap."
)

print(
    "PASS: The three evaluation periods "
    "are separated."
)

,period,start,end,stride,number_of_origins,first_origin,last_origin
0,development,2024-01-03,2024-12-18,5,51,2024-01-03,2024-12-18
1,inner_validation,2025-01-22,2025-06-25,5,23,2025-01-22,2025-06-25
2,outer_validation,2025-07-30,2025-12-17,5,21,2025-07-30,2025-12-17


PASS: The three evaluation periods are separated.


Define the LightGBM agent assignment

In [14]:
LIVE_AGENT_PROMPT = f"""
Tune {METHOD} for the {HORIZON}-business-day BAA10Y forecast using
the {COVARIATE_PANEL} covariate panel and its matching fixed benchmark.

Run {INITIAL_TRIALS} broad trials, review paired diagnostics, choose
an evidence-based focus action, and continue to {TOTAL_TRIALS} total
trials. Freeze only a robust, non-overfitting candidate that improves
both development and inner-validation CRPS.

Do not access outer validation, eval_2026, or promote a model.
""".strip()


print(LIVE_AGENT_PROMPT)

Tune lightgbm for the 5-business-day BAA10Y forecast using
the default covariate panel and its matching fixed benchmark.

Run 5 broad trials, review paired diagnostics, choose
an evidence-based focus action, and continue to 10 total
trials. Freeze only a robust, non-overfitting candidate that improves
both development and inner-validation CRPS.

Do not access outer validation, eval_2026, or promote a model.


In [ ]:
async def run_live_agent_with_trace(
    prompt: str,
):
    """Run the adaptive agent and record every tool call."""

    from aieng.forecasting.methods.agentic import (
        build_adk_agent,
    )
    from google.adk.runners import (
        InMemoryRunner,
    )
    from google.genai import (
        types as genai_types,
    )

    agent_config = (
        build_baa10y_adaptive_config(
            state_path=STATE_PATH,
            max_output_tokens=4096,
        )
    )

    live_agent = build_adk_agent(
        agent_config
    )

    app_name = (
        "baa10y_lightgbm_adaptive_test"
    )

    user_id = "notebook04_user"

    runner = InMemoryRunner(
        agent=live_agent,
        app_name=app_name,
    )

    session = await (
        runner.session_service
        .create_session(
            app_name=app_name,
            user_id=user_id,
        )
    )

    message = genai_types.Content(
        role="user",
        parts=[
            genai_types.Part(
                text=prompt
            ),
        ],
    )

    tool_trace = []
    response_parts = []

    try:
        async for event in runner.run_async(
            user_id=user_id,
            session_id=session.id,
            new_message=message,
        ):
            content = getattr(
                event,
                "content",
                None,
            )

            parts = (
                getattr(
                    content,
                    "parts",
                    None,
                )
                or []
            )

            for part in parts:
                function_call = getattr(
                    part,
                    "function_call",
                    None,
                )

                if function_call is not None:
                    arguments = (
                        getattr(
                            function_call,
                            "args",
                            None,
                        )
                        or {}
                    )

                    tool_trace.append({
                        "tool": (
                            function_call.name
                        ),
                        "arguments": dict(
                            arguments
                        ),
                    })

                text_value = getattr(
                    part,
                    "text",
                    None,
                )

                if (
                    event.is_final_response()
                    and text_value
                ):
                    response_parts.append(
                        text_value
                    )

    finally:
        await runner.close()

    return (
        "\n".join(response_parts),
        tool_trace,
    )


(
    LIVE_AGENT_REPLY,
    LIVE_AGENT_TRACE,
) = await run_live_agent_with_trace(
    LIVE_AGENT_PROMPT
)


print("Agent response:")
print()

print(
    LIVE_AGENT_REPLY
    or "No final text response returned."
)

print(
    "\nNumber of tool calls:",
    len(LIVE_AGENT_TRACE),
)

In [ ]:
from pathlib import Path

import pandas as pd
import yaml

from aieng.forecasting.evaluation import (
    MultiTargetBacktestSpec,
)


spec_path = (
    ROOT
    / "implementations"
    / "BAA10Y_forecasting"
    / "specs"
    / "baa10y_validate_2025.yaml"
)

with spec_path.open(
    encoding="utf-8"
) as file:
    tuning_spec = (
        MultiTargetBacktestSpec.model_validate(
            yaml.safe_load(file)
        )
    )

origin_dates = pd.date_range(
    start=tuning_spec.start,
    end=tuning_spec.end,
    freq="B",
)[::tuning_spec.stride]
origin_check = pd.DataFrame({
    "origin": origin_dates,
    "weekday": origin_dates.day_name(),
})

display(
    origin_check
)

assert (
    origin_check["weekday"]
    == "Wednesday"
).all()

print(
    "PASS: All tuning origins are Wednesdays."
)

,origin,weekday
0,2025-07-30,Wednesday
1,2025-08-06,Wednesday
2,2025-08-13,Wednesday
3,2025-08-20,Wednesday
4,2025-08-27,Wednesday
5,2025-09-03,Wednesday
6,2025-09-10,Wednesday
7,2025-09-17,Wednesday
8,2025-09-24,Wednesday
9,2025-10-01,Wednesday


PASS: All tuning origins are Wednesdays.
